In [0]:
%sql
--drop table sales_catalog.silver.s_crm_cust_info;
--drop table sales_catalog.silver.s_crm_prd_info;

select * from  sales_catalog.silver.s_crm_prd_info limit 2

In [0]:
%sql

-- crm_cust_info
INSERT INTO sales_catalog.silver.s_crm_cust_info
(cst_id, cst_key, cst_firstname, cst_lastname, cst_marital_status, cst_gndr, cst_create_date)

WITH silver_transformation AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) AS flag_last
    FROM
        sales_catalog.bronze.b_crm_cust_info
        WHERE cst_id IS NOT NULL
)
SELECT
    cst_id,
    cst_key,
    TRIM(cst_firstname) AS cst_firstname,
    TRIM(cst_lastname) AS cst_lastname,
    CASE 
        WHEN UPPER(TRIM(cst_marital_status)) = 'S' THEN 'Single'
        WHEN UPPER(TRIM(cst_marital_status)) = 'M' THEN 'Married'
        ELSE 'Unknown'
    END AS cst_marital_status,
    CASE 
        WHEN UPPER(TRIM(cst_gndr)) = 'F' THEN 'Female'
        WHEN UPPER(TRIM(cst_gndr)) = 'M' THEN 'Male'
        ELSE 'Unknown'
    END AS cst_gndr,
    cst_create_date
FROM
    silver_transformation
WHERE
    flag_last = 1

In [0]:
%sql

-- crm_prd_info
INSERT INTO sales_catalog.silver.s_crm_prd_info
(prd_id, cat_id, prd_key, prd_nm, prd_cost, prd_line, prd_start_dt, prd_end_dt)
SELECT 
    prd_id, 
    REPLACE(SUBSTRING(prd_key, 1, 5), '-', '_') AS cat_id,
    SUBSTRING(prd_key, 7) AS prd_key,
    prd_nm,
    COALESCE(prd_cost, 0) AS prd_cost,
    CASE 
        WHEN UPPER(TRIM(prd_line)) = 'M' THEN 'Mountain'
        WHEN UPPER(TRIM(prd_line)) = 'R' THEN 'Road'
        WHEN UPPER(TRIM(prd_line)) = 'S' THEN 'Other Sales'
        WHEN UPPER(TRIM(prd_line)) = 'T' THEN 'Tour'
        ELSE 'Unknown'
    END AS prd_line,
    prd_start_dt,
    LEAD(prd_start_dt) OVER (
        PARTITION BY prd_key 
        ORDER BY prd_start_dt ASC
    ) - 1 AS prd_end_dt
FROM 
    sales_catalog.bronze.b_crm_prd_info
WHERE 
    prd_id IS NOT NULL

In [0]:
%sql

INSERT INTO sales_catalog.silver.s_sales_details
(sls_ord_num, sls_prd_key, sls_cust_id, sls_order_dt, sls_ship_dt, sls_due_dt, sls_sales, sls_quantity, sls_price)
SELECT 
    sls_ord_num,
    sls_prd_key,
    sls_cust_id,
    CASE 
        WHEN sls_order_dt = 0 OR LENGTH(CAST(sls_order_dt AS STRING)) != 8 THEN NULL 
        ELSE TO_DATE(CAST(sls_order_dt AS STRING), 'yyyyMMdd') 
    END AS sls_order_dt,
    CASE 
        WHEN sls_ship_dt = 0 OR LENGTH(CAST(sls_ship_dt AS STRING)) != 8 THEN NULL 
        ELSE TO_DATE(CAST(sls_ship_dt AS STRING), 'yyyyMMdd') 
    END AS sls_ship_dt,
    CASE 
        WHEN sls_due_dt = 0 OR LENGTH(CAST(sls_due_dt AS STRING)) != 8 THEN NULL 
        ELSE TO_DATE(CAST(sls_due_dt AS STRING), 'yyyyMMdd') 
    END AS sls_due_dt,
    CASE 
        WHEN sls_sales IS NULL OR sls_sales <= 0 OR sls_sales != sls_quantity * ABS(sls_price) THEN sls_quantity * ABS(sls_price) 
        ELSE sls_sales
    END AS sls_sales,
    sls_quantity,
    CASE 
        WHEN sls_price IS NULL OR sls_price <= 0 THEN sls_sales / COALESCE(sls_quantity, 0) 
        ELSE sls_price 
    END AS sls_price
FROM sales_catalog.bronze.b_sales_details

In [0]:
%sql

INSERT INTO sales_catalog.silver.s_cust_az12
    (CID, BDATE, GEN)
SELECT 
    CASE 
        WHEN CID LIKE 'NAS%' THEN SUBSTRING(CID, 4)
        ELSE CID 
    END AS CID, 
    CASE 
        WHEN BDATE > CURRENT_DATE() THEN NULL 
        ELSE BDATE
    END AS BDATE, 
    CASE 
        WHEN UPPER(TRIM(GEN)) IN ('F', 'FEMALE') THEN 'Female'
        WHEN UPPER(TRIM(GEN)) IN ('M', 'MALE') THEN 'Male'
        ELSE 'Unknown'
    END AS GEN
FROM 
    sales_catalog.bronze.b_cust_az12

In [0]:
%sql
select * from sales_catalog.silver.s_cust_az12 limit 3

In [0]:
%sql

INSERT INTO sales_catalog.silver.s_loc_a101_cntry
(CID, CNTRY)
SELECT 
    REPLACE(CID, '-', '') AS CID,
    CASE 
        WHEN TRIM(CNTRY) = 'DE' THEN 'Germany'
        WHEN TRIM(CNTRY) IN ('US', 'USA') THEN 'United States'
        WHEN TRIM(CNTRY) = '' OR CNTRY IS NULL THEN 'Unknown'
        ELSE TRIM(CNTRY) 
    END AS CNTRY
FROM 
    sales_catalog.bronze.b_loc_a101_cntry 

In [0]:
%sql

INSERT INTO sales_catalog.silver.s_px_cat_g1v2
(ID, CAT, SUBCAT, MAINTENANCE)
select 
ID,
CAT,
SUBCAT,
MAINTENANCE
from sales_catalog.bronze.b_px_cat_g1v2